# Exercise: consume_02 — consumer groups & manual offsets

**Goal:** understand how Kafka tracks *which messages have been processed* and what happens when consumers crash mid-message.

**Prerequisite:** there is data in `strom` (run `exercise_01_produce_single.ipynb` or `_02`).

## Background — auto-commit vs. manual commit

By default the consumer auto-commits its offset every 5 seconds. That's convenient but risky:
if the process *processes* a message and *crashes before* the auto-commit, the offset
advances anyway and the message is lost (At-Most-Once).

With `enable.auto.commit: False` we control when to commit — **after** successful processing.

```
read message → process (DB write, API call) → commit offset
```

If the processing crashes, the offset is never committed → on restart the message is re-delivered. This is **At-Least-Once** delivery.

## Step 1 — read with manual commit

Run the cell. Notice the `committed` step printed after each message.

In [ ]:
from confluent_kafka import Consumer

consumer = Consumer({
    'bootstrap.servers': 'redpanda:29092',
    'group.id':           'manual-commit-demo',
    'auto.offset.reset':  'earliest',
    'enable.auto.commit': False,
})
consumer.subscribe(['strom'])

print(f'{"#":>3} | {"Key":>7} | {"Offset":>6} | Status')
print('-' * 45)

messages_read, empty_polls = 0, 0
while messages_read < 5 and empty_polls < 5:
    msg = consumer.poll(2.0)
    if msg is None: empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0; messages_read += 1

    key = msg.key().decode() if msg.key() else 'None'
    print(f'{messages_read:>3} | {key:>7} | {msg.offset():>6} | read — processing...', end='')
    # ...imagine a DB insert here. If it FAILS we never commit.
    consumer.commit(message=msg)
    print(' committed.')

consumer.close()
print(f'\nProcessed and committed {messages_read} messages.')

## Task A — simulate a crash

Modify the cell above so it **does not** call `consumer.commit(...)`. Run it once, then run it again with the same `group.id`. Do you see the same messages a second time? Why?

> Hint: without commits, Kafka thinks the messages were never processed → re-delivery on the next read.

In [ ]:
# TODO: copy the cell above and remove the consumer.commit(message=msg) line.
# Run twice with the same group.id and observe.


## Task B — two consumers in the same group

1. Open this notebook in **two browser tabs** (right-click the file → *Open in New Tab*).
2. In *both* tabs, run the cell below — same `group.id`.
3. In a *third* tab, run `exercise_03_produce_batch.ipynb` to send a batch.
4. **Watch:** which consumer gets which events? Why?

> Within a consumer group each partition is owned by exactly one consumer. With 1 partition this is unbalanced; with 2 partitions both consumers each get one.

In [ ]:
from confluent_kafka import Consumer
from datetime import datetime

consumer = Consumer({
    'bootstrap.servers': 'redpanda:29092',
    'group.id':          'group-experiment',
    'auto.offset.reset': 'latest',
})
consumer.subscribe(['strom', 'wasser'])
print('Listening — start the producer in another tab. Stop with the ■ button.')

try:
    while True:
        msg = consumer.poll(0.5)
        if msg is None or msg.error(): continue
        ts = datetime.now().strftime('%H:%M:%S')
        key = msg.key().decode() if msg.key() else 'None'
        print(f'[{ts}] {msg.topic()} P{msg.partition()} offset={msg.offset()} key={key}')
except KeyboardInterrupt:
    print('\nStopped.')
finally:
    consumer.close()